In [ ]:
from bng_simulator.utils.logger_utils import load_log_data
import numpy as np

# Make matplotlib plots interactive (allows zoom/pan, etc.)
%matplotlib widget
import matplotlib.pyplot as plt

In [ ]:
# Run number
run = 2
metadata, data = load_log_data(run)

In [ ]:
metadata.keys()

In [ ]:
data.keys()

In [ ]:
log_data = data[('ego', 'gtstate')]
log_data = {k: np.array(v) for k, v in log_data.items()}

In [ ]:
for k, v in log_data.items():
    print(f"{k}: {v.shape}")

In [ ]:
def plot_data(data, x_key="time", y_keys=None, title="", figsize=(12, 8)):
    """
    Plots several data keys as a function of another key (generally time).
    
    Args:
        data (dict): Dictionary containing data arrays.
        x_key (str): Key to use for the x-axis (default is "time").
        y_keys (list, optional): List of keys to plot on the y-axis. 
            If None, all keys except x_key will be used.
        title (str, optional): Title for the entire figure.
        figsize (tuple, optional): Size of the figure.
    
    Raises:
        ValueError: If no valid y_keys are found.
    """
    # Determine which keys to plot if not explicitly provided.
    if y_keys is None:
        y_keys = [k for k in data.keys() if k != x_key]
    
    n_plots = len(y_keys)
    if n_plots == 0:
        raise ValueError("No y_keys provided or found in data.")

    # Decide a suitable grid based on the number of plots.
    ncols = int(np.ceil(np.sqrt(n_plots)))
    nrows = int(np.ceil(n_plots / ncols))
    
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False,sharex=True)
    
    # Plot each y_key against the x_key.
    for idx, key in enumerate(y_keys):
        row = idx // ncols
        col = idx % ncols
        ax = axs[row][col]
        ax.plot(data[x_key], data[key])
        ax.set_xlabel(x_key)
        ax.set_ylabel(key)
        ax.set_title(key)
    
    # Hide any unused subplots.
    for idx in range(n_plots, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        axs[row][col].set_visible(False)
    
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Let's plot just the time to see if monotonic.
plt.plot(log_data['time'])

In [ ]:
dt = np.diff(log_data['time'])
print(f"Min dt: {np.min(dt)}")
print(f"Max dt: {np.max(dt)}")
print(f"Mean dt: {np.mean(dt)}")

In [ ]:
# Plot steering angle
plot_data(log_data, y_keys=['steering', 'wheelFL_angle', 'wheelFR_angle'], figsize=(8, 4))

In [ ]:
# Let's define the data for fitting steering geometry
time_arr = log_data['time']
steering = log_data['steering']
wheelFL_angle = log_data['wheelFL_angle']
wheelFR_angle = log_data['wheelFR_angle']
wheel_angle = 0.5 * (wheelFL_angle + wheelFR_angle)

In [ ]:
# Should we filter the data according to some metrics?
valid_index = np.logical_and(time_arr >= 371, time_arr <= 386)
wheel_angle = wheel_angle[valid_index]
steering = steering[valid_index]
wheelFL_angle = wheelFL_angle[valid_index]

In [ ]:
# A scipy based approach to fit the steering
# a represents the wheel angle, b the steering angle, and c is wheelFL_angle
from scipy.optimize import least_squares
def residuals(p, a, b, c):
    """
    Compute residuals for the steering geometry model.
    p = [x, y, z] -> x - steering ratio, y - static toe angle, z - ackerman ratio (full term)
    a : wheel angle (average of FL and FR)
    b : steering angle
    c : wheelFL angle
    """
    x, y, z = p
    # Compute model predictions for each data point
    pred_a = x * b
    pred_c = -y + (pred_a) + z * (pred_a)**2
    # Residuals for both equations
    res1 = pred_a - a
    res2 = pred_c - c
    # Concatenate the residuals from both equations
    return np.concatenate([res1, res2])

In [ ]:
# Run it
initial_guess = [1., 0, 0.01] # gearratio, static toe, ackerman ratio
# Run the least squares optimization with a robust loss function.
# "soft_l1" is a good default choice to reduce the influence of outliers.
result = least_squares(
    residuals, 
    initial_guess,
    args=(wheel_angle, steering, wheelFL_angle),
    loss='soft_l1' # what else can we use? 'cauchy', 'arctan', 'huber', 'linear', 'soft_l1'
)

# Optimized parameters
x_opt, y_opt, z_opt = result.x
print("Optimized parameters:", result.x)

# Estimate the covariance matrix of the parameters.
# result.cost is 0.5 * sum(residuals**2), so we scale accordingly.
J = result.jac
dof = len(result.fun) - len(result.x)  # degrees of freedom
cov = np.linalg.inv(J.T @ J) * (2 * result.cost / dof)
std_devs = np.sqrt(np.diag(cov))
print("Parameter uncertainties (approx.):", std_devs)

# Let's pack the results into a dictionary
fit_results = {
    'gear_ratio': x_opt,
    'static_toe': y_opt,
    'ackerman_ratio': z_opt,
    'gear_ratio_uncertainty': std_devs[0],
    'static_toe_uncertainty': std_devs[1],
    'ackerman_ratio_uncertainty': std_devs[2]
}

In [ ]:
# Let's try a more robust approach to fitting the steering geometry
# Using mcmc
import jax.numpy as jnp
from jax import random
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

_init_guess = [1., 0, 0.01]
def model(a, b, c):
    """
    p = [x, y, z] -> x - steering ratio, y - static toe angle, z - ackerman ratio (full term)
    a : wheel angle (average of FL and FR)
    b : steering angle
    c : wheelFL angle
    """
    # Define weakly informative priors for the parameters.
    # These priors work well in many cases without much tuning.
    gear_ratio = numpyro.sample('gear_ratio', dist.Normal(_init_guess[0], 5))
    static_toe = numpyro.sample('static_toe', dist.Normal(_init_guess[1], 5))
    ackerman_ratio = numpyro.sample('ackerman_ratio', dist.Normal(_init_guess[2], 5))
    
    # Priors for the noise scales (using half-Cauchy for positive values).
    sigma_a = numpyro.sample('sigma_a', dist.HalfCauchy(5))
    sigma_c = numpyro.sample('sigma_c', dist.HalfCauchy(5))
    
    # Compute model predictions for each data point
    pred_a = gear_ratio * b
    pred_c = -static_toe + (pred_a) + ackerman_ratio * (pred_a)**2
    
    # Likelihoods for the observed data
    numpyro.sample('a_obs', dist.Normal(pred_a, sigma_a), obs=a)
    numpyro.sample('c_obs', dist.Normal(pred_c, sigma_c), obs=c)

In [ ]:
# Convert the data to JAX arrays
wheel_angle_jax = jnp.array(wheel_angle)
steering_jax = jnp.array(steering)
wheelFL_angle_jax = jnp.array(wheelFL_angle)

# Set up and run MCMC using the NUTS sampler.
# Here we choose 500 warm-up steps and 1000 samples.
# These numbers are fairly standard, and you can adjust them if needed.
rng_key = random.PRNGKey(10)
nuts_kernel = NUTS(model)
mcmc = MCMC(nuts_kernel, num_warmup=500, num_samples=1000)
mcmc.run(rng_key, a=wheel_angle_jax, b=steering_jax, c=wheelFL_angle_jax)
mcmc.print_summary()

In [ ]:
# Sample the MCMC chains to get the parameter estimates.
# Print the mean and 95% credible interval for each parameter.
# Plot the distributions of the parameters.

# Get the MCMC samples
samples = mcmc.get_samples()
fit_results_mcmc = {
    'gear_ratio': np.mean(samples['gear_ratio']),
    'static_toe': np.mean(samples['static_toe']),
    'ackerman_ratio': np.mean(samples['ackerman_ratio']),
    'gear_ratio_uncertainty': np.std(samples['gear_ratio']),
    'static_toe_uncertainty': np.std(samples['static_toe']),
    'ackerman_ratio_uncertainty': np.std(samples['ackerman_ratio'])
}

# Plot the MCMC samples
plt.figure(figsize=(8, 4))
plt.hist(samples['gear_ratio'], bins=30, alpha=0.5, label='gear_ratio')
plt.hist(samples['static_toe'], bins=30, alpha=0.5, label='static_toe')
plt.hist(samples['ackerman_ratio'], bins=30, alpha=0.5, label='ackerman_ratio')
plt.xlabel('Parameter value')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
print ("MCMC results: ")
fit_results_mcmc

In [ ]:
print ("lsq results: ")
fit_results

In [ ]:
import jax

# Let's try to estimate steering dynamics as a function of input.
def simulate_steering_model(K, tau, L, t, u, delta0):
    """
    Simulate the first-order system with a pure delay over a given time sequence.
    
    The continuous-time model is:
        τ * dδ/dt + δ(t) = K * u(t - L)
    which we rewrite as:
        dδ/dt = (K * u(t - L) - δ(t)) / τ
    
    Using a simple Euler integration, we have:
        δ[k+1] = δ[k] + dt * (K * u(t_k - L) - δ[k]) / τ
    
    Parameters:
      K     : scalar, system gain.
      tau   : scalar, time constant (must be > 0).
      L     : scalar, pure delay (≥ 0).
      t     : 1D JAX array of time points (assumed sorted in ascending order).
      u     : 1D JAX array of control input values, same length as t.
      delta0: initial output value δ(t₀). Default is 0.
      
    Returns:
      A 1D JAX array with the simulated output δ(t) at each time in t.
    """
    def step_fn(delta, i):
        # Compute the time step dt
        dt = t[i + 1] - t[i]
        # Compute the effective time at which to evaluate the input:
        effective_t = t[i] - L
        # Use linear interpolation to get u(effective_t).
        # jnp.interp returns u[0] if effective_t is before t[0] and u[-1] if after t[-1].
        u_eff = jnp.interp(effective_t, t, u)
        # Compute the derivative dδ/dt
        ddelta_dt = (K * u_eff - delta) / tau
        # Euler update
        delta_next = delta + dt * ddelta_dt
        return delta_next, delta_next

    # Loop over all time steps using jax.lax.scan for efficiency.
    indices = jnp.arange(t.shape[0] - 1)
    _, delta_steps = jax.lax.scan(step_fn, delta0, indices)
    # Prepend the initial condition
    simulated_output = jnp.concatenate([jnp.array([delta0]), delta_steps])
    return simulated_output

In [ ]:
# Vectorize simulate model.
vmap_simulate_steering_model = jax.vmap(simulate_steering_model, in_axes=(None, None, None, 0, 0, 0))
jit_vmap_simulate_steering_model = jax.jit(vmap_simulate_steering_model)
def bayesian_steering_model(t, u, delta_obs):
    # Priors on the parameters:
    K = numpyro.sample("K", dist.Normal(1.0, 5))
    tau = numpyro.sample("tau", dist.HalfNormal(1))
    L = numpyro.sample("L", dist.HalfNormal(1))
    
    # prior for the noise scale
    sigma = numpyro.sample("sigma", dist.HalfNormal(5))
    
    # delta_obs is a sequence of points of size t + 1
    pred_delta = jit_vmap_simulate_steering_model(K, tau, L, t, u, delta_obs[...,0])
    
    # Likelihood - assume independent Gaussian noise between observed and predicted delta
    numpyro.sample("delta_obs", dist.Normal(pred_delta[...,1:], sigma), obs=delta_obs[...,1:])
    

In [ ]:
# Let's try to fit the steering dynamics
steering_input = log_data['steeringInput']
steering_output = log_data['steering']
time_steering = log_data['time']

# indexes
valid_index = np.logical_and(time_steering >= 371, time_steering <= 386)
steering_input = steering_input[valid_index]
steering_output = steering_output[valid_index]
time_steering = time_steering[valid_index]

# Size of the fitting horizon
H_fit = 50 # around 1 second

# Split the data into 2d where second dimension is the fitting horizon
time_steering_2d = jnp.array([time_steering[i:i+H_fit+1] for i in range(len(time_steering) - H_fit)])[30:40]
steering_input_2d = jnp.array([steering_input[i:i+H_fit+1] for i in range(len(steering_input) - H_fit)])[30:40]
# The output should have one more element than the input
steering_output_2d = jnp.array([steering_output[i:i+H_fit+1] for i in range(len(steering_output) - H_fit)])[30:40]

# Set up and run MCMC using the NUTS sampler.
rng_key = random.PRNGKey(1)
# vectorize the model
steering_nuts_kernel = NUTS(bayesian_steering_model)
steering_mcmc = MCMC(steering_nuts_kernel, num_warmup=200, num_samples=500)
steering_mcmc.run(rng_key, t=time_steering_2d, u=steering_input_2d, delta_obs=steering_output_2d)
steering_mcmc.print_summary()



In [ ]:
# Let's plot the results of the MCMC fitting
# Get the MCMC samples
steering_samples = steering_mcmc.get_samples()
fit_results_steering_mcmc = {
    'K': np.mean(steering_samples['K']),
    'tau': np.mean(steering_samples['tau']),
    'L': np.mean(steering_samples['L']),
    'sigma': np.mean(steering_samples['sigma']),
    'K_uncertainty': np.std(steering_samples['K']),
    'tau_uncertainty': np.std(steering_samples['tau']),
    'L_uncertainty': np.std(steering_samples['L']),
    'sigma_uncertainty': np.std(steering_samples['sigma'])
}

# Plot the MCMC samples
plt.figure(figsize=(8, 4))
plt.hist(steering_samples['K'], bins=30, alpha=0.5, label='K')
plt.hist(steering_samples['tau'], bins=30, alpha=0.5, label='tau')
plt.hist(steering_samples['L'], bins=30, alpha=0.5, label='L')
plt.hist(steering_samples['sigma'], bins=30, alpha=0.5, label='sigma')


In [ ]:
fit_results_steering_mcmc


In [ ]:
steering_input = log_data['steeringInput']
steering_output = log_data['steering']
time_steering = log_data['time'] - log_data['time'][0]

# Let's do some check/.
max_steering = np.max(steering_output)
scaled_steering = steering_output / max_steering

# Let;s plot the scaled steering and steering input on same plot
plt.figure(figsize=(8, 4))
plt.plot(time_steering, steering_input * -1, label='steering input')
plt.plot(time_steering, scaled_steering, label='scaled steering output')
plt.xlabel('Time (s)')
plt.ylabel('Steering')
plt.legend()
plt.show()
